# Lab 11 · Reference solution

The polished final implementation of [Lab 11: Generator-critic from scratch](../README.md).

Extends Lab 10's supervisor + researcher + writer with a critic worker and a bounded refinement loop. `MAX_REFINEMENT_CYCLES = 3` caps the loop; the cap fires explicitly with `ok_with_unresolved_issues`, not a forced approval.

This notebook is the reference implementation; refer to [`../lab.ipynb`](../lab.ipynb) for the pedagogical step-by-step build. The [`solution README`](./README.md) covers implementation choices, common variations, and bugs to watch for.

## Setup

In [ ]:
import hashlib
import json
import os
import pathlib
import re
import warnings
from dataclasses import dataclass, field
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, ConfigDict, Field

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)

PROVIDER = "openai"
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]
print(f"Using {PROVIDER} / {MODEL}")


## Lab 10 machinery (chat client, web tools, researcher, writer)

Unchanged from Lab 10's solution. Repeated here for self-containment.

In [ ]:
@dataclass
class ToolCall:
    id: str
    name: str
    arguments: dict


@dataclass
class AssistantMessage:
    content: str | None
    tool_calls: list[ToolCall] = field(default_factory=list)


def chat_with_tools(messages: list[dict], tools: list[dict] | None = None,
                     tool_choice: str = "auto", temperature: float = 0) -> AssistantMessage:
    if PROVIDER == "openai":
        from openai import OpenAI
        resp = OpenAI().chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
            tool_choice=tool_choice if tools else None, temperature=temperature,
        )
        msg = resp.choices[0].message
        return AssistantMessage(
            content=msg.content,
            tool_calls=[
                ToolCall(id=tc.id, name=tc.function.name,
                         arguments=json.loads(tc.function.arguments))
                for tc in (msg.tool_calls or [])
            ],
        )
    elif PROVIDER == "anthropic":
        from anthropic import Anthropic
        client = Anthropic()
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        non_system = [m for m in messages if m["role"] != "system"]
        anth_tools = [
            {"name": t["function"]["name"], "description": t["function"]["description"],
             "input_schema": t["function"]["parameters"]}
            for t in (tools or [])
        ]
        resp = client.messages.create(
            model=MODEL, system=system, messages=non_system,
            tools=anth_tools or None, max_tokens=2048, temperature=temperature,
        )
        text = "".join(b.text for b in resp.content if hasattr(b, "text"))
        tcs = [ToolCall(id=b.id, name=b.name, arguments=dict(b.input))
               for b in resp.content if getattr(b, "type", None) == "tool_use"]
        return AssistantMessage(content=text or None, tool_calls=tcs)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def _action_hash(name: str, args: dict) -> str:
    return hashlib.sha256(
        (name + "|" + json.dumps(args, sort_keys=True)).encode()
    ).hexdigest()[:16]


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


### Web tools (unchanged from Lab 10)

In [ ]:
from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException, TimeoutException
import requests
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.simplefilter("ignore", MarkupResemblesLocatorWarning)

RecencyType = Literal["any", "day", "week", "month", "year"]
_RECENCY_MAP: dict[str, str | None] = {
    "any": None, "day": "d", "week": "w", "month": "m", "year": "y",
}
USER_AGENT = ("AgenticAIEngineer-CourseLab/0.1 "
              "(https://github.com/MHHamdan/Agentic-AI-Engineer)")
PAYWALL_MARKERS = [
    "subscribe to read", "subscribe to continue",
    "create a free account to continue",
    "you've reached your free article limit", "register to read",
]


def web_search(query: str, recency: RecencyType = "any", max_results: int = 8) -> dict:
    if not query or not query.strip():
        return {"status": "error", "kind": "other", "detail": "empty query"}
    try:
        with DDGS(timeout=15) as ddgs:
            raw = ddgs.text(query=query.strip(), region="us-en", safesearch="moderate",
                            timelimit=_RECENCY_MAP.get(recency),
                            max_results=max_results, backend="auto")
    except RatelimitException as e:
        return {"status": "error", "kind": "rate_limit", "detail": str(e)}
    except TimeoutException as e:
        return {"status": "error", "kind": "timeout", "detail": str(e)}
    except DDGSException as e:
        return {"status": "error", "kind": "other", "detail": str(e)}
    except Exception as e:
        return {"status": "error", "kind": "other", "detail": f"{type(e).__name__}: {e}"}
    if not raw:
        return {"status": "empty", "query": query, "detail": "no results"}
    return {"status": "ok",
            "results": [{"title": (r.get("title") or "").strip(),
                         "url": (r.get("href") or "").strip(),
                         "snippet": (r.get("body") or "").strip()}
                        for r in raw if r.get("href")][:max_results]}


def fetch_page(url: str, max_chars: int = 8000) -> dict:
    if not url or not url.startswith(("http://", "https://")):
        return {"status": "error", "url": url, "kind": "other", "detail": "invalid url"}
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT},
                            timeout=15, allow_redirects=True)
    except requests.Timeout:
        return {"status": "error", "url": url, "kind": "timeout",
                "detail": "request timed out after 15s"}
    except requests.RequestException as e:
        return {"status": "error", "url": url, "kind": "other",
                "detail": f"{type(e).__name__}: {e}"}
    if 400 <= resp.status_code < 500:
        kind = "blocked" if resp.status_code in (401, 403, 429) else "http_4xx"
        return {"status": "error", "url": url, "kind": kind,
                "detail": f"HTTP {resp.status_code}"}
    if 500 <= resp.status_code < 600:
        return {"status": "error", "url": url, "kind": "http_5xx",
                "detail": f"HTTP {resp.status_code}"}
    try:
        soup = BeautifulSoup(resp.text, "html.parser")
    except Exception as e:
        return {"status": "error", "url": url, "kind": "parse",
                "detail": f"{type(e).__name__}: {e}"}
    for tag in soup(["script", "style", "nav", "footer", "aside",
                     "header", "form", "iframe", "noscript"]):
        tag.decompose()
    title = (soup.title.string.strip() if soup.title and soup.title.string else "")
    text = re.sub(r"\n{3,}", "\n\n", soup.get_text(separator="\n")).strip()
    if any(m in text[:5000].lower() for m in PAYWALL_MARKERS):
        return {"status": "error", "url": url, "kind": "paywall",
                "detail": "paywall markers detected"}
    if len(text) > max_chars:
        return {"status": "too_long", "url": url, "title": title,
                "text": text[:max_chars], "total_chars": len(text)}
    return {"status": "ok", "url": url, "title": title, "text": text}


### Researcher and writer workers (unchanged from Lab 10)

In [ ]:
WORKER_MAX_STEPS = 8

RESEARCHER_TOOLS = [
    {"type": "function",
     "function": {"name": "web_search",
                  "description": "Search the web.",
                  "parameters": {"type": "object",
                                 "properties": {"query": {"type": "string"},
                                                "recency": {"type": "string",
                                                            "enum": ["any", "day", "week", "month", "year"]},
                                                "max_results": {"type": "integer"}},
                                 "required": ["query"]}}},
    {"type": "function",
     "function": {"name": "fetch_page",
                  "description": "Fetch a URL's text content.",
                  "parameters": {"type": "object",
                                 "properties": {"url": {"type": "string"},
                                                "max_chars": {"type": "integer"}},
                                 "required": ["url"]}}},
]


def _researcher_execute(name: str, args: dict) -> dict:
    if name == "web_search":
        return web_search(args.get("query", ""), args.get("recency", "any"),
                          args.get("max_results", 8))
    if name == "fetch_page":
        return fetch_page(args.get("url", ""), args.get("max_chars", 8000))
    return {"status": "error", "kind": "unknown_tool", "detail": name}


RESEARCHER_SYSTEM_PROMPT = """You are a researcher worker. You receive ONE question.
Search the web, fetch 1-3 most relevant pages, then emit your FINAL response as JSON:

  {"findings": "<2-4 sentences with [1], [2] inline citations>",
   "citations": [{"url": "<url>", "title": "<title>"}, ...]}

Rules:
- Cite by [1], [2] inline; citations list maps in order.
- Do NOT call the same tool with the same args twice.
- Do NOT produce prose outside the JSON envelope.
"""


def researcher_agent(question: str) -> dict:
    messages: list[dict] = [
        {"role": "system", "content": RESEARCHER_SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    seen: set[str] = set()
    fetches: list[dict] = []
    for _step in range(WORKER_MAX_STEPS):
        msg = chat_with_tools(messages, tools=RESEARCHER_TOOLS)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)
        if not msg.tool_calls:
            try:
                raw = (msg.content or "").strip()
                if raw.startswith("```"):
                    raw = raw.strip("`").split("\n", 1)[1].rstrip("`").strip()
                obj = json.loads(raw)
                return {"status": "ok",
                        "findings": obj.get("findings", ""),
                        "citations": obj.get("citations", [])}
            except (json.JSONDecodeError, IndexError):
                return {"status": "error", "kind": "bad_envelope",
                        "detail": "researcher returned non-JSON final"}
        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": "you already called this with same args"}
            else:
                seen.add(ah)
                tool_result = _researcher_execute(tc.name, tc.arguments)
                if tc.name == "fetch_page" and tool_result.get("status") in ("ok", "too_long"):
                    fetches.append({"url": tool_result["url"],
                                    "title": tool_result.get("title", "")})
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:4000]})
    partial = f"Research did not complete within {WORKER_MAX_STEPS} steps."
    if fetches:
        partial += f" Fetched {len(fetches)} page(s) without producing a final summary."
    return {"status": "step_cap", "findings": partial, "citations": fetches}


WRITER_SYSTEM_PROMPT = """You are a writer worker. You receive a brief containing findings,
citations, and (optionally) revision issues from a previous critique. Produce ~150 words
of prose that:

1. States the findings accurately. Do not invent claims.
2. Preserves citations: inline [1], [2], etc., then list at the end as:
       [1] Title — URL
3. If revision issues are provided, address each one specifically.
4. If brief_status='step_cap', say so explicitly.

Return ONLY the prose. No JSON wrapping.
"""


def writer_agent(findings: str, citations: list[dict], brief_status: str = "ok",
                  revision_issues: list[dict] | None = None) -> dict:
    citation_lines = "\n".join(
        f"[{i + 1}] {c.get('title', '?')} — {c.get('url', '?')}"
        for i, c in enumerate(citations)
    )
    user_prompt = (
        f"BRIEF (status: {brief_status}):\n\n"
        f"FINDINGS:\n{findings}\n\n"
        f"CITATIONS:\n{citation_lines or '(none)'}\n\n"
    )
    if revision_issues:
        issues_block = "\n".join(
            f"- [{i['kind']}] {i['detail']}" for i in revision_issues
        )
        user_prompt += f"REVISION REQUESTED — address each issue:\n{issues_block}\n\n"
    user_prompt += "Compose the prose."
    msg = chat_with_tools(
        [{"role": "system", "content": WRITER_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    return {"status": "ok", "prose": (msg.content or "").strip()}


## Critic worker

Takes `(findings, citations, draft)`. Returns `{status: "ok"}` or `{status: "needs_revision", issues: [{kind, detail}]}`. Five enumerated `kind` values. Capped at 3 issues per call. Defaults to OK on borderline cases.

In [ ]:
CRITIC_ISSUE_KINDS = [
    "unsupported_claim",   # claim in draft doesn't appear in findings
    "missing_citation",    # claim should cite a source but doesn't
    "dropped_citation",    # findings cite a source the draft doesn't
    "unclear_prose",       # phrasing materially obscures meaning
    "format_violation",    # missing citation list, wrong inline format, etc.
]


CRITIC_SYSTEM_PROMPT = """You are a STRICT reviewer. You receive a brief (findings,
citations) and a draft of prose composed from that brief.

Apply these checks IN ORDER:

1. unsupported_claim — for each factual claim in the draft, find a sentence in
   findings that supports it. If none, flag.
2. missing_citation — claims that draw on findings but don't cite a source.
3. dropped_citation — any citation in the brief NOT used in the draft.
4. format_violation — missing the [1] [2] inline references, missing the
   citation list at the end, or wrong format ("Source 1:" instead of "[1]").
5. unclear_prose — only flag if phrasing materially obscures meaning. Don't
   flag mere style preferences.

RULES:
- Default to OK on borderline cases. When uncertain, return ok.
- Cap at 3 issues. More than 3 → structural problems, not point fixes.
- Quote the specific draft passage that fails the check.
- Return ONLY valid JSON. No markdown fences, no prose preamble.

If draft passes all checks:
   {"status": "ok"}

If checks fail:
   {"status": "needs_revision", "issues": [
     {"kind": "<one of CRITIC_ISSUE_KINDS>", "detail": "<quote + explanation>"},
     ...
   ]}
"""


def critic_agent(findings: str, citations: list[dict], draft: str) -> dict:
    citation_lines = "\n".join(
        f"[{i + 1}] {c.get('title', '?')} — {c.get('url', '?')}"
        for i, c in enumerate(citations)
    )
    user_prompt = (
        f"BRIEF:\n\n"
        f"FINDINGS:\n{findings}\n\n"
        f"CITATIONS:\n{citation_lines or '(none)'}\n\n"
        f"DRAFT TO REVIEW:\n{draft}\n\n"
        f"Apply the 5 checks. Return JSON only."
    )
    msg = chat_with_tools(
        [{"role": "system", "content": CRITIC_SYSTEM_PROMPT},
         {"role": "user", "content": user_prompt}],
        temperature=0,
    )
    raw = (msg.content or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if "\n" in raw:
            raw = raw.split("\n", 1)[1]
        if raw.endswith("```"):
            raw = raw[:-3]
        raw = raw.strip()
    try:
        result = json.loads(raw)
    except json.JSONDecodeError:
        # Parse failure → assume OK (avoid blocking on critic errors)
        return {"status": "ok", "_parse_warning": raw[:200]}
    if result.get("status") not in ("ok", "needs_revision"):
        return {"status": "ok", "_invalid_status": result.get("status")}
    if result["status"] == "needs_revision":
        issues = result.get("issues", []) or []
        valid = [i for i in issues
                 if isinstance(i, dict) and i.get("kind") in CRITIC_ISSUE_KINDS][:3]
        if not valid:
            return {"status": "ok", "_no_valid_issues": True}
        return {"status": "needs_revision", "issues": valid}
    return {"status": "ok"}


## Supervisor with bounded refinement

Step cap raised to 10 (vs Lab 10's 6) to accommodate the refinement loop. Three worker-call tools. The refinement loop is inside the supervisor's tool dispatch — when critic returns `needs_revision`, the supervisor re-calls the writer with the issues attached.

When `cycles == MAX_REFINEMENT_CYCLES` and the critic still returns `needs_revision`, the supervisor surfaces `ok_with_unresolved_issues` rather than forcing approval.

In [ ]:
SUPERVISOR_MAX_STEPS = 10
MAX_REFINEMENT_CYCLES = 3


class CallResearcherArgs(StrictModel):
    question: str = Field(description="Question for the researcher.")


class CallWriterAndCriticArgs(StrictModel):
    findings: str = Field(description="Researcher's findings.")
    citations: list[dict] = Field(default_factory=list,
                                   description="Researcher's citations.")
    brief_status: str = Field(default="ok",
                               description="Researcher's status: 'ok' or 'step_cap'.")


def _call_researcher_tool(args: CallResearcherArgs) -> dict:
    return researcher_agent(args.question)


def _call_writer_critic_loop(args: CallWriterAndCriticArgs) -> dict:
    """Run the writer → critic refinement loop up to MAX_REFINEMENT_CYCLES.

    Composes writer + critic + bounded refinement as a single supervisor-
    facing tool. This is cleaner than exposing call_writer and call_critic
    separately — keeps the refinement-cycle bookkeeping out of the
    supervisor's reasoning surface.
    """
    revision_issues: list[dict] = []
    last_draft = ""
    last_critic: dict = {"status": "ok"}

    for cycle in range(MAX_REFINEMENT_CYCLES + 1):
        writer_result = writer_agent(
            args.findings, args.citations, args.brief_status,
            revision_issues=revision_issues if revision_issues else None,
        )
        last_draft = writer_result["prose"]
        last_critic = critic_agent(args.findings, args.citations, last_draft)

        if last_critic["status"] == "ok":
            return {"status": "ok", "prose": last_draft, "cycles": cycle,
                    "final_critic_status": "ok"}

        if cycle == MAX_REFINEMENT_CYCLES:
            # Cap fired: surface unresolved issues honestly. NOT a forced 'ok'.
            return {"status": "ok_with_unresolved_issues",
                    "prose": last_draft, "cycles": cycle,
                    "unresolved_issues": last_critic.get("issues", [])}

        revision_issues = last_critic.get("issues", [])

    # Should be unreachable, but defend against logic bugs
    return {"status": "ok_with_unresolved_issues",
            "prose": last_draft, "cycles": MAX_REFINEMENT_CYCLES,
            "unresolved_issues": last_critic.get("issues", [])}


SUPERVISOR_TOOLS_REGISTRY: dict = {
    "call_researcher": (
        _call_researcher_tool, CallResearcherArgs,
        "Dispatch a question to the researcher. Returns {status, findings, citations}.",
    ),
    "call_writer_critic_loop": (
        _call_writer_critic_loop, CallWriterAndCriticArgs,
        "Dispatch the brief to the writer-critic refinement loop. The loop writes "
        "a draft, critiques it, revises if needed, up to MAX_REFINEMENT_CYCLES=3. "
        "Returns {status, prose, cycles, ...}. status='ok' means clean approval; "
        "'ok_with_unresolved_issues' means cap fired — surface honestly to the user.",
    ),
}


def _supervisor_schemas() -> list[dict]:
    return [
        {"type": "function",
         "function": {"name": name, "description": desc,
                      "parameters": args_model.model_json_schema()}}
        for name, (_fn, args_model, desc) in SUPERVISOR_TOOLS_REGISTRY.items()
    ]


def _supervisor_dispatch(call: ToolCall) -> dict:
    if call.name not in SUPERVISOR_TOOLS_REGISTRY:
        return {"status": "error", "kind": "unknown_worker", "tool": call.name,
                "available": list(SUPERVISOR_TOOLS_REGISTRY)}
    fn, args_model, _ = SUPERVISOR_TOOLS_REGISTRY[call.name]
    try:
        return fn(args_model.model_validate(call.arguments))
    except Exception as e:
        return {"status": "error", "kind": "supervisor_dispatch_error",
                "detail": f"{type(e).__name__}: {e}"}


SUPERVISOR_SYSTEM_PROMPT = """You are a supervisor agent. You coordinate workers via
two tool calls: call_researcher and call_writer_critic_loop.

WORKFLOW:
1. call_researcher with the user's question.
2. Read the researcher's envelope. If status='step_cap', still proceed.
3. call_writer_critic_loop with the brief. The loop runs writer → critic →
   maybe-revise up to 3 times.
4. Read the loop's return envelope:
   - status='ok': return the prose as your final answer.
   - status='ok_with_unresolved_issues': return the prose, but include a brief
     prefix like "(Note: review cycle hit cap with unresolved issues.)" before
     the prose. Be honest with the user.

RULES:
- Do not call the same worker twice with the same args.
- Pass citations VERBATIM to the writer-critic loop.
- If a worker returns an error envelope, surface the error.
"""


def supervisor_agent(task: str) -> dict:
    messages: list[dict] = [
        {"role": "system", "content": SUPERVISOR_SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    seen_actions: set[str] = set()
    schemas = _supervisor_schemas()
    refinement_cycles_used = 0
    final_loop_status: str | None = None

    for _step in range(SUPERVISOR_MAX_STEPS):
        msg = chat_with_tools(messages, tools=schemas)
        entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            entry["tool_calls"] = [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.name, "arguments": json.dumps(tc.arguments)}}
                for tc in msg.tool_calls
            ]
        messages.append(entry)

        if not msg.tool_calls:
            return {"status": "ok", "answer": msg.content or "",
                    "steps_used": _step + 1,
                    "refinement_cycles_used": refinement_cycles_used,
                    "final_loop_status": final_loop_status}

        for tc in msg.tool_calls:
            ah = _action_hash(tc.name, tc.arguments)
            if ah in seen_actions:
                tool_result = {"status": "error", "kind": "repeated_action",
                               "detail": f"You already called {tc.name} with these args."}
            else:
                seen_actions.add(ah)
                tool_result = _supervisor_dispatch(tc)
                if tc.name == "call_writer_critic_loop":
                    refinement_cycles_used = tool_result.get("cycles", 0)
                    final_loop_status = tool_result.get("status")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(tool_result)[:6000]})

    return {"status": "step_cap", "answer": "[supervisor hit step cap]",
            "steps_used": SUPERVISOR_MAX_STEPS,
            "refinement_cycles_used": refinement_cycles_used,
            "final_loop_status": final_loop_status}


## Demo

In [ ]:
task = (
    "Research recent developments in the Model Context Protocol (MCP) "
    "and write a 150-word summary with citations."
)
result = supervisor_agent(task)
print(f"Status: {result['status']}, supervisor steps: {result['steps_used']}, "
      f"refinement cycles: {result['refinement_cycles_used']}, "
      f"loop status: {result['final_loop_status']}")
print("=" * 70)
print(result["answer"])


**Sample output (LLM responses will vary; trajectory should be stable):**

```
Status: ok, supervisor steps: 3, refinement cycles: 0, loop status: ok
======================================================================
The Model Context Protocol (MCP) is an open standard introduced by Anthropic
for connecting AI agents to external data sources and tools [1]. Recent
developments include expanded server libraries [2] and production deployments
at major companies [3]...

[1] Introducing the Model Context Protocol — https://www.anthropic.com/news/...
[2] MCP Server Gallery — https://...
[3] ...
```

A typical clean run takes 0 refinement cycles: the writer's first draft passes the critic's checks. Runs that surface revision are equally valid — they show the cycle doing its job. The `final_loop_status` field distinguishes clean approval (`ok`) from cap-fire (`ok_with_unresolved_issues`).

## Production readiness — out of scope here

For a real deployment you'd also want: per-cycle logging (which issues the critic flagged each cycle, which the writer addressed), eval harnesses that score `unresolved_issues` against a labeled benchmark, A/B testing of critic temperature, prompt-version tracking (sycophancy emerges when a previously-strict critic gets a softer rewrite), and trace IDs spanning the refinement loop. The cap-firing path needs first-class handling downstream — alerting, escalation to a human reviewer, separate metrics from the clean-approval path.

The next pattern (Lab 12) takes a different shape: upfront plan emission, bounded parallel execution. The chat client, action-hash dedup, structured envelopes, and StrictModel patterns carry over; the refinement loop is replaced by a different control-flow primitive (plan dispatch + replanning).